# ScamRoad — Autonomous Cloud GPU Training, Model Comparison & Optimization Pipeline

**Repository**: [github.com/Priya-Ranjan-0201/potholes](https://github.com/Priya-Ranjan-0201/potholes)  
**Compute Target**: Google Colab GPU (Tesla T4 / V100 / A100 / L4 with CUDA)  
**Persistent Cloud Hub**: Google Drive (`/content/drive/MyDrive/ScamRoad`)  
**Architectures Trained & Compared**:
1. **YOLO11n (Nano)** — Next-gen ultra-lightweight mobile detector (2.6M params, <10ms)
2. **YOLO11s (Small)** — Next-gen high-recall road defect detector (9.4M params)
3. **YOLO11n-seg (Segmentation)** — Instance segmentation for pixel-level damage surface area
4. **RT-DETR-l (Transformer)** — Real-time end-to-end detection transformer (NMS-free)
5. **YOLOv8n (Baseline)** — Reference edge baseline model (3.2M params)
**Single-Source Taxonomy**: `pothole`, `crack_longitudinal`, `crack_transverse`, `crack_alligator`


## Step 1: Hardware & System Resource Audit
Verify the assigned GPU, VRAM, and system memory before launching any heavy operations.

In [ ]:
# 1. GPU & System Resource Audit
!nvidia-smi

import psutil
import shutil
import torch

ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print('=' * 65)
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device       : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM Capacity    : {vram:.2f} GB')
    print(f'CUDA Version     : {torch.version.cuda}')
else:
    print('[WARNING] No CUDA GPU detected! Switch Runtime -> Change runtime type -> T4 GPU')
print(f'Colab RAM Total  : {ram.total / (1024**3):.2f} GB (Available: {ram.available / (1024**3):.2f} GB)')
print(f'Colab Disk Free  : {disk.free / (1024**3):.2f} GB / {disk.total / (1024**3):.2f} GB')
print('=' * 65)


## Step 2: Mount Persistent Google Drive Storage
Mount Google Drive to persist checkpoints, trained weights, logs, and exports across Colab disconnects.

In [ ]:
# 2. Google Drive Mount & Workspace Directory Initialization
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/ScamRoad')
drive_root.mkdir(parents=True, exist_ok=True)

for folder in ['datasets', 'checkpoints', 'weights', 'exports', 'runs', 'logs', 'comparisons']:
    (drive_root / folder).mkdir(exist_ok=True)

print(f'[SUCCESS] Persistent Google Drive storage active at: {drive_root}')
!ls -lh /content/drive/MyDrive/ScamRoad


## Step 3: Clone or Pull ScamRoad Project Repository
Pulls the latest verified codebase from GitHub repository `Priya-Ranjan-0201/potholes`.

In [ ]:
# 3. Clone or pull repo
import os, sys
from pathlib import Path

repo_path = Path('/content/potholes')
if not repo_path.exists():
    !git clone https://github.com/Priya-Ranjan-0201/potholes.git /content/potholes
else:
    !cd /content/potholes && git pull origin main

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

# Set working directory to repo root
os.chdir(str(repo_path))
print('[SUCCESS] Project workspace ready at:', repo_path.resolve())


## Step 4: Install Cloud Dependencies
Install Ultralytics YOLOv8, ONNX, and OpenCV acceleration libraries.

In [ ]:
# 4. Pip installation
!pip install -q ultralytics onnx onnxruntime-gpu pyyaml albumentations opencv-python psutil


## Step 5: Configure Kaggle API Credentials
Authenticates Kaggle directly from Colab Secrets (`priyaranjanng`, `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`).

In [ ]:
# 5. Kaggle Authentication
import contextlib, json, os
from pathlib import Path

def clean_token(raw_token):
    if not raw_token:
        return None
    token = raw_token.strip()
    if (token.startswith('export ') and '=' in token) or ('=' in token and not token.startswith('{')):
        token = token.split('=', 1)[1].strip()
    return token.strip('"\'\r\n ')

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_file = kaggle_dir / 'kaggle.json'
access_token_file = kaggle_dir / 'access_token'

try:
    from google.colab import userdata
    user = None
    raw_key = None
    with contextlib.suppress(Exception): user = userdata.get('KAGGLE_USERNAME')
    with contextlib.suppress(Exception): raw_key = userdata.get('KAGGLE_KEY')
    with contextlib.suppress(Exception):
        if not raw_key: raw_key = userdata.get('KAGGLE_API_TOKEN')
    for candidate in ['priyaranjanng', 'priyaranja', 'priyaranjan', 'kaggle']:
        if not raw_key:
            with contextlib.suppress(Exception): raw_key = userdata.get(candidate)
    key = clean_token(raw_key)
    username = user or 'priyaranjanng'
    if key:
        access_token_file.write_text(key, encoding='utf-8')
        access_token_file.chmod(0o600)
        kaggle_file.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
        kaggle_file.chmod(0o600)
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = key
        os.environ['KAGGLE_API_TOKEN'] = key
        print(f'[SUCCESS] Kaggle credentials configured from Colab Secrets (user: {username})!')
except Exception as e:
    print('[INFO] Colab secrets check:', e)

!kaggle --version


## Step 6: Dataset Download & Ingestion Pipeline
Downloads road damage datasets (MakeML Potholes and RDD2022) using Kaggle API,
converts annotations to standard YOLO format, audits class balances, and generates leakage-free sequence splits.

In [ ]:
# 6. Download Road Defect Datasets from Kaggle
import shutil
from pathlib import Path

data_raw = Path('/content/data/raw')
data_raw.mkdir(parents=True, exist_ok=True)

# Download MakeML Potholes
!kaggle datasets download -d larxel/pothole-detection -p /content/data/raw/potholes --unzip

# Convert VOC annotations to normalized YOLO format
!python -m ml.data.convert --format voc --xml-dir /content/data/raw/potholes/annotations --image-dir /content/data/raw/potholes/images --output-dir /content/data/processed/potholes/labels

# Stage images into processed directory
proc_img = Path('/content/data/processed/potholes/images')
proc_img.mkdir(parents=True, exist_ok=True)
for img in Path('/content/data/raw/potholes/images').glob('*.*'):
    if img.is_file():
        shutil.copy2(img, proc_img / img.name)

# Audit dataset (pHash deduplication, corrupt checks, class distributions)
!python -m ml.data.audit --image-dir /content/data/processed/potholes/images --label-dir /content/data/processed/potholes/labels

# Split into 70/15/15 sequence-aware splits
!python -m ml.data.split --image-dir /content/data/processed/potholes/images --label-dir /content/data/processed/potholes/labels --output-dir /content/data/splits --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15

# Generate all 7 standard manifests and audit deliverables in Google Drive
!python -m ml.data.clean_and_unify --data-dir /content/data/splits --output-dir /content/drive/MyDrive/ScamRoad/dataset_artifacts

print('[SUCCESS] Dataset ready for model training and manifests stored in Drive.')


[INFO] Attempting download from Kaggle: larxel/pothole-detection...
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata

You don't have access to this resource.
If this is a competition, you may need to accept its rules first at
  https://www.kaggle.com/competitions/<competition>/rules
[INFO] Attempting download from Kaggle: aliabdelmenam/rdd-2022...
Dataset URL: https://www.kaggle.com/datasets/aliabdelmenam/rdd-2022
License(s): CC-BY-SA-4.0
100% 9.90G/9.90G [01:55<00:00, 92.4MB/s]

[SUCCESS] Acquired 38385 images from aliabdelmenam/rdd-2022!
Converting VOC annotations from /content/data/raw/potholes/annotations to /content/data/processed/potholes/labels...
Conversion complete! Report saved to ml/runs/audit/conversion_summary.json
Auditing images in /content/data/processed/potholes/images...
Auditing labels in /content/data/processed/potholes/labels...
Audit complete! Output saved to ml/runs/audit/audit_latest.json and docs/DATAS

## Step 7: Train Model 1 — YOLO11n (Next-Gen Nano Detector)
Ultra-compact architecture (2.6M params, 6.5 GFLOPs) optimized for real-time mobile surveying on Android.


In [22]:
# 7. Train YOLO11n (Nano)
from ultralytics import YOLO
from pathlib import Path

drive_runs = Path('/content/drive/MyDrive/ScamRoad/runs')
dataset_cfg = Path('/content/data/splits/dataset.yaml')

# Fallback to yolov8n if yolo11 weights are unavailable in current runtime
base_weight = 'yolo11n.pt'
try:
    model_11n = YOLO(base_weight)
except Exception:
    base_weight = 'yolov8n.pt'
    model_11n = YOLO(base_weight)

print(f'[INFO] Training {base_weight} on cloud GPU...')
results_11n = model_11n.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(drive_runs),
    name='model_yolo11n',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLO11n training finished. Checkpoints saved to Google Drive.')


[INFO] Training yolo11n.pt on cloud GPU...
Ultralytics 8.4.198 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/splits/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=model_yolo11n

RuntimeError: Dataset '/content/data/splits/dataset.yaml' error ❌ '/content/data/splits/dataset.yaml' does not exist

## Step 8: Train Model 2 — YOLO11s (Next-Gen Small Detector)
Enhanced feature pyramid capacity (9.4M params) for superior recall on fine longitudinal and transverse cracks.


In [ ]:
# 8. Train YOLO11s (Small)
base_s = 'yolo11s.pt'
try:
    model_11s = YOLO(base_s)
except Exception:
    base_s = 'yolov8s.pt'
    model_11s = YOLO(base_s)

print(f'[INFO] Training {base_s} on cloud GPU...')
results_11s = model_11s.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(drive_runs),
    name='model_yolo11s',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLO11s training finished.')


## Step 9: Train Model 3 — YOLO11n-seg (Instance Segmentation)
Predicts pixel-level defect masks alongside bounding boxes for calibrated relative defect surface area.


In [ ]:
# 9. Train YOLO11n-seg
base_seg = 'yolo11n-seg.pt'
try:
    model_seg = YOLO(base_seg)
except Exception:
    base_seg = 'yolov8n-seg.pt'
    model_seg = YOLO(base_seg)

print(f'[INFO] Training {base_seg} instance segmentation on cloud GPU...')
results_seg = model_seg.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(drive_runs),
    name='model_yolo11n_seg',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLO11n-seg training finished.')


## Step 10: Train Model D — RT-DETR-l (Real-Time Transformer)
End-to-end detection transformer removing NMS thresholds and excelling on dense crack clusters.

In [ ]:
# 10. Train RT-DETR-l
model_rtdetr = YOLO('rtdetr-l.pt')
results_rtdetr = model_rtdetr.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project=str(drive_runs),
    name='model_rtdetr_l',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] RT-DETR-l training finished.')


## Step 11: Multi-Model Architecture Comparison & Benchmark
Evaluates all trained architectures on the identical validation split,
profiles latency (mean, P95, FPS), model size, and generates a structured Markdown comparison table.


In [ ]:
# 11. Run Multi-Model Comparison Harness
comparison_dir = Path('/content/drive/MyDrive/ScamRoad/comparisons')

!python -m ml.evaluation.compare_models \
    --data /content/data/splits/dataset.yaml \
    --models \
        YOLO11n=/content/drive/MyDrive/ScamRoad/runs/model_yolo11n/weights/best.pt \
        YOLO11s=/content/drive/MyDrive/ScamRoad/runs/model_yolo11s/weights/best.pt \
        YOLO11n-seg=/content/drive/MyDrive/ScamRoad/runs/model_yolo11n_seg/weights/best.pt \
        RT-DETR-l=/content/drive/MyDrive/ScamRoad/runs/model_rtdetr_l/weights/best.pt \
    --out-dir /content/drive/MyDrive/ScamRoad/comparisons \
    --device 0

# Display Markdown Comparison Report
from IPython.display import Markdown, display
report_file = comparison_dir / 'model_comparison.md'
if report_file.exists():
    display(Markdown(report_file.read_text()))


## Step 12: ONNX INT8 Quantization & Mobile Edge Export
Exports the top-performing model to ONNX INT8 for edge deployment on Android and low-power SBCs.

In [ ]:
# 12. Export & Quantize Best Model
import shutil
from pathlib import Path
from ultralytics import YOLO
from onnxruntime.quantization import quantize_dynamic, QuantType

drive_runs = Path('/content/drive/MyDrive/ScamRoad/runs')
exports_dir = Path('/content/drive/MyDrive/ScamRoad/exports')
exports_dir.mkdir(parents=True, exist_ok=True)

# Dynamically locate top trained model
candidate_models = [
    drive_runs / 'model_yolo11n' / 'weights' / 'best.pt',
    drive_runs / 'model_yolo11s' / 'weights' / 'best.pt',
    drive_runs / 'model_yolo11n_seg' / 'weights' / 'best.pt',
    drive_runs / 'model_rtdetr_l' / 'weights' / 'best.pt',
    drive_runs / 'model_yolov8n' / 'weights' / 'best.pt',
]
best_pt = next((p for p in candidate_models if p.exists()), None)

if best_pt:
    print(f'[INFO] Exporting and quantizing top model: {best_pt}')
    best_model = YOLO(str(best_pt))
    onnx_file = best_model.export(format='onnx', imgsz=640, simplify=True)
    fp32_dest = exports_dir / 'scamroad_best_fp32.onnx'
    shutil.copy(onnx_file, fp32_dest)

    int8_dest = exports_dir / 'scamroad_best_int8.onnx'
    quantize_dynamic(
        model_input=str(fp32_dest),
        model_output=str(int8_dest),
        weight_type=QuantType.QUInt8
    )
    print(f'[SUCCESS] FP32 ONNX : {fp32_dest} ({fp32_dest.stat().st_size / (1024**2):.2f} MB)')
    print(f'[SUCCESS] INT8 ONNX : {int8_dest} ({int8_dest.stat().st_size / (1024**2):.2f} MB)')
else:
    print('[WARNING] No trained best.pt found in runs directory to export.')


## Step 13: Static Image & Real-Time Video Defect Inference
Runs `scripts/infer_media.py` on test images and video with bounding boxes, HUD overlay, and relative damage area %.

In [ ]:
# 13. Test Image & Video Inference
from pathlib import Path

drive_runs = Path('/content/drive/MyDrive/ScamRoad/runs')
candidate_models = [
    drive_runs / 'model_yolo11n' / 'weights' / 'best.pt',
    drive_runs / 'model_yolo11s' / 'weights' / 'best.pt',
    drive_runs / 'model_yolo11n_seg' / 'weights' / 'best.pt',
    drive_runs / 'model_rtdetr_l' / 'weights' / 'best.pt',
    drive_runs / 'model_yolov8n' / 'weights' / 'best.pt',
]
selected_weight = next((str(p) for p in candidate_models if p.exists()), 'yolo11n.pt')
print(f'[INFO] Running inference with weights: {selected_weight}')

!python scripts/infer_media.py \
    --source /content/data/splits/images/val \
    --weights "{selected_weight}" \
    --output-dir /content/drive/MyDrive/ScamRoad/inference_demo \
    --conf 0.25

print('[SUCCESS] Inference demonstration complete. Results saved in Google Drive.')
!ls -lh /content/drive/MyDrive/ScamRoad/inference_demo


## Step 14: Google Drive Persistent Inventory
Lists all finalized weights, ONNX models, benchmark reports, and inference media stored in Google Drive.

In [ ]:
# 14. Artifact Inventory
!ls -R /content/drive/MyDrive/ScamRoad
print('\n[COMPLETE] ScamRoad Cloud GPU Pipeline completed successfully.')
